# Compute and plot vertical state-metric cross-sections

Process 3-D variables into latitude–level or longitude–level sections, cache the requested diagnostics, and plot prior and posterior sections.

In [ ]:
# =============================================================================
# TOP-LEVEL PATH PARAMETERS
# Edit these three paths to relocate inputs, the analysis checkout, or outputs.
# =============================================================================
from pathlib import Path
import sys

INPUT_DATA_ROOT = Path("/compyfs/zhan391/v3_dart_cda_scratch")
WORK_DIR = Path("/compyfs/zhan391/work_package/e3sm_dart_analysis")
DIAGNOSTIC_OUTPUT_ROOT = Path("/compyfs/www/zhan391/e3sm_dart/diag_dart_2026")
WORKFLOW_NAME = "analysis_atm_init"

In [ ]:
# Derived paths: built from the parameters above.
import sys
from pathlib import Path

DIAGNOSTIC_DIR = WORK_DIR / "diagnostic"
if not DIAGNOSTIC_DIR.is_dir():
    raise FileNotFoundError(f"Diagnostic work directory not found: {DIAGNOSTIC_DIR}")
if str(DIAGNOSTIC_DIR) not in sys.path:
    sys.path.insert(0, str(DIAGNOSTIC_DIR))

from configs.output_paths import workflow_output_dirs
WORKFLOW_DATA_DIR, WORKFLOW_FIGURE_DIR = workflow_output_dirs(
    WORKFLOW_NAME,
    output_root=DIAGNOSTIC_OUTPUT_ROOT,
)

## Notes: vertical cross-section variables and metrics

Cross-sections are available for the 3-D variables `U`, `V`, `T`, `Q`,
`CLDLIQ`, and `CLDICE`.

Available metrics:

| Metric | Meaning |
|---|---|
| `bias` | Prior/posterior minus reference, averaged across the collapsed horizontal direction |
| `rmse` | Prior/posterior RMSE against reference across the collapsed horizontal direction |
| `spread` | Prior/posterior ensemble standard deviation averaged across the collapsed direction |

Set `horizontal_axis` to `"latitude"` for a latitude–level section (longitude
is averaged) or `"longitude"` for a longitude–level section (latitude is
averaged). `LAT_RANGE` and `LON_RANGE` define both the displayed and averaged
region. Request-keyed NetCDF caches are reused unless `force_compute = True`.


## Selection

In [ ]:
ANALYSIS_DATA_DIR = WORKFLOW_DATA_DIR / "remapped_ensemble_diagnostics"
CACHE_DIR = WORKFLOW_DATA_DIR / "atmosphere_metrics"
FIGURE_DIR = WORKFLOW_FIGURE_DIR / "vertical_cross_sections"

EXPERIMENTS = ["CAM6-S0", "DART10-S0", "DART20-S0", "DART40-S0"]
REFERENCE_EXPERIMENT = "CAM6-S0"
TIMESTAMPS = [
    "2011-12-26-00000",
    "2011-12-26-21600",
    "2011-12-26-43200",
    "2011-12-26-64800",
]

CROSS_SECTION_REQUESTS = [
    {"variable": "T", "metric": "rmse", "horizontal_axis": "latitude"},
    # {"variable": "Q", "metric": "spread", "horizontal_axis": "longitude"},
]

LAT_RANGE = (-90, 90)
LON_RANGE = (-180, 180)
force_compute = False
RUN_WORKFLOW = True

THREE_D_VARIABLES = {"U", "V", "T", "Q", "CLDLIQ", "CLDICE"}
CROSS_SECTION_METRICS = {"bias", "rmse", "spread"}
HORIZONTAL_AXES = {"latitude", "longitude"}

MODEL_LABELS = {
    "CAM6-S0": "CAM6-DART (ENS=80)",
    "CTRL10-S0": "CTRL (ENS=10)",
    "DART10-S0": "EAM-DART (ENS=10)",
    "DART20-S0": "EAM-DART (ENS=20)",
    "DART40-S0": "EAM-DART (ENS=40)",
    "DART40INF0p6-S0": "EAM-DART (ENS=40, INF=0.6)",
}


## Imports

In [ ]:
import matplotlib.pyplot as plt
import xarray as xr

from util.atm_initialization_metrics import (
    DartAssimOutput,
    cache_cross_section_metric,
    cross_section_cache_path,
    plot_cached_cross_sections,
)


## 1. Read raw data and process/cache requested metrics

In [ ]:
if RUN_WORKFLOW:
    if REFERENCE_EXPERIMENT not in EXPERIMENTS:
        raise ValueError("REFERENCE_EXPERIMENT must be included in EXPERIMENTS")
    for request in CROSS_SECTION_REQUESTS:
        if request["variable"] not in THREE_D_VARIABLES:
            raise ValueError(f"Cross-sections require a 3-D variable: {request}")
        if request["metric"] not in CROSS_SECTION_METRICS:
            raise ValueError(f"Unsupported cross-section metric: {request}")
        if request["horizontal_axis"] not in HORIZONTAL_AXES:
            raise ValueError(f"Unsupported horizontal axis: {request}")

    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    for timestamp in TIMESTAMPS:
        paths = {
            (experiment, request["variable"], request["metric"],
             request["horizontal_axis"]): cross_section_cache_path(
                CACHE_DIR, experiment, timestamp, request["variable"],
                request["metric"], request["horizontal_axis"]
            )
            for experiment in EXPERIMENTS
            for request in CROSS_SECTION_REQUESTS
        }
        if not force_compute and all(path.exists() for path in paths.values()):
            print(f"{timestamp}: all cross-sections exist; reusing cache.")
            continue

        print(f"{timestamp}: deriving missing/forced cross-sections.")
        outputs = {
            experiment: DartAssimOutput.from_experiment(
                ANALYSIS_DATA_DIR, experiment, timestamp
            )
            for experiment in EXPERIMENTS
        }
        reference = outputs[REFERENCE_EXPERIMENT].post_mean
        try:
            for (experiment, variable, metric, axis), path in paths.items():
                dataset = cache_cross_section_metric(
                    path, outputs[experiment], reference, variable, metric,
                    LAT_RANGE, LON_RANGE, horizontal_axis=axis,
                    force_compute=force_compute,
                )
                dataset.close()
        finally:
            for output in outputs.values():
                for dataset in (
                    output.prior_mean, output.prior_sd,
                    output.post_mean, output.post_sd,
                ):
                    dataset.close()


## 2. Plot cached metrics

In [ ]:
if RUN_WORKFLOW:
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)
    for timestamp in TIMESTAMPS:
        for request in CROSS_SECTION_REQUESTS:
            variable = request["variable"]
            metric = request["metric"]
            axis = request["horizontal_axis"]
            paths = {
                experiment: cross_section_cache_path(
                    CACHE_DIR, experiment, timestamp, variable, metric, axis
                )
                for experiment in EXPERIMENTS
            }
            missing = [path for path in paths.values() if not path.is_file()]
            if missing:
                raise FileNotFoundError(
                    "Missing cross-section caches:\n" +
                    "\n".join(str(path) for path in missing)
                )
            cached = {
                experiment: xr.load_dataset(path)
                for experiment, path in paths.items()
            }
            try:
                figure = plot_cached_cross_sections(
                    cached, variable, metric, label_map=MODEL_LABELS,
                    title=f"{variable} {metric.upper()} {axis}-level section - {timestamp}",
                )
                figure.savefig(
                    FIGURE_DIR / f"{variable}_{metric}_{axis}_{timestamp}.png",
                    dpi=200, bbox_inches="tight",
                )
                plt.close(figure)
            finally:
                for dataset in cached.values():
                    dataset.close()
